# Tworzenie aplikacji generujących obrazy (OpenAI)

LLM-y to nie tylko generowanie tekstu. Możesz także generować obrazy na podstawie opisów tekstowych. Obrazy jako modalność są użyteczne w MedTech, architekturze, turystyce, tworzeniu gier, marketingu i innych dziedzinach. W tej lekcji pracujemy z dzisiejszymi modelami **GPT Image** na platformie OpenAI.

## Cele nauki

Na koniec tej lekcji będziesz potrafić:

- Wyjaśnić, czym jest generowanie obrazów i gdzie jest użyteczne.
- Zrozumieć rodzinę modeli `gpt-image` oraz jak różnią się od starszych modeli DALL·E.
- Zbudować aplikację do generowania obrazów i edycji obrazów.

## Czym jest generowanie obrazów?

Modele generujące obrazy tworzą obrazy na podstawie tekstowego polecenia (promptu). Nowoczesne modele takie jak `gpt-image` uczą się relacji między tekstem a obrazami podczas treningu, a następnie iteracyjnie przekształcają losowy szum w obraz odpowiadający twojemu promptowi.

Dwie dobrze znane rodziny modeli obrazów to:

- **`gpt-image` (OpenAI)** — obecna generacja używana w tej lekcji. Wspiera generowanie obrazów z tekstu oraz edycję obrazów (inpainting z maską).
- **Midjourney** — popularny model firm trzecich z własną usługą i workflow opartym na Discordzie.

> Starsze modele obrazów OpenAI — **DALL·E 2** i **DALL·E 3** — to modele przestarzałe. DALL·E 3 nie jest już dostępny dla nowych wdrożeń, a funkcje takie jak `create_variation` istniały tylko w DALL·E 2. Do nowych aplikacji używaj modeli `gpt-image`.

> **Ważne:** modele `gpt-image` zwracają wygenerowany obraz jako **base64** (`b64_json`), a nie jako URL. Twój kod dekoduje ciąg base64 do bajtów i zapisuje — nie ma adresu URL z obrazem do pobrania.


## Tworzenie pierwszej aplikacji do generowania obrazów

Co jest potrzebne do stworzenia aplikacji do generowania obrazów? Potrzebujesz następujących bibliotek:

- **python-dotenv**, zdecydowanie zalecane jest użycie tej biblioteki, aby przechowywać swoje sekrety w pliku *.env* z dala od kodu.
- **openai**, tej biblioteki będziesz używać do komunikacji z API OpenAI.
- **pillow**, do pracy z obrazami w Pythonie.


1. Utwórz plik *.env* z następującą zawartością:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Zbierz powyższe biblioteki w pliku o nazwie *requirements.txt* w następujący sposób:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Następnie utwórz środowisko wirtualne i zainstaluj biblioteki:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> W systemie Windows użyj następujących poleceń, aby utworzyć i aktywować swoje wirtualne środowisko:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Dodaj następujący kod do pliku o nazwie *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Utwórz obiekt OpenAI (odczytuje OPENAI_API_KEY z pliku .env)
    client = openai.OpenAI()


    try:
        # Utwórz obraz za pomocą API do generowania obrazów
        generation_response = client.images.generate(
            model="gpt-image-1",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Wprowadź tutaj tekst swojego zapytania
            size='1024x1024',
            n=1
        )
        # Ustaw katalog do przechowywania obrazu
        image_dir = os.path.join(os.curdir, 'images')

        # Jeśli katalog nie istnieje, utwórz go
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Zainicjuj ścieżkę obrazu (uwaga: rozszerzenie pliku powinno być png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # Modele gpt-image zwracają obraz jako base64 (b64_json), a nie URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Wyświetl obraz w domyślnej przeglądarce obrazów
        image = Image.open(image_path)
        image.show()

    # przechwyć wyjątki
    except openai.BadRequestError as err:
        print(err)

    ```

Wyjaśnijmy ten kod:

- Najpierw importujemy potrzebne biblioteki, w tym bibliotekę OpenAI, dotenv, moduł base64 oraz bibliotekę Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- Następnie tworzymy klienta, który odczytuje klucz API z twojego pliku ``.env``.

    ```python
    # Utwórz obiekt OpenAI
    client = openai.OpenAI()
    ```

- Kolejnym krokiem generujemy obraz:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-1",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Wpisz tutaj tekst swojego polecenia
        size='1024x1024',
        n=1
    )
    ```

    Modele `gpt-image` zwracają obraz jako łańcuch **base64** w `data[0].b64_json`. Dekodujemy go do bajtów i zapisujemy do pliku — nie ma adresu URL do pobrania.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Na koniec otwieramy obraz i wyświetlamy go za pomocą standardowej przeglądarki obrazów:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Więcej szczegółów na temat generowania obrazu

Przyjrzyjmy się parametrom `images.generate`:

- **model**, to model obrazu do użycia (na przykład `gpt-image-1`).
- **prompt**, to tekstowy opis użyty do wygenerowania obrazu. Tutaj jest to "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size**, to rozmiar wygenerowanego obrazu (na przykład `1024x1024`, `1536x1024`, `1024x1536` lub `"auto"`).
- **n**, to liczba generowanych obrazów. Tutaj generujemy jeden.

> Modele obrazów nie przyjmują parametru `temperature` — to kontrola generowania tekstu. Aby uzyskać różnorodność, wywołuj API ponownie; aby zmniejszyć różnorodność, sprecyzuj swój opis.

## Dodatkowe możliwości generowania obrazów

Widziałeś, jak wygenerować obraz za pomocą kilku linii Pythona. Modele `gpt-image` mogą także **edytować** istniejący obraz. Podając obraz, opcjonalną **maseczkę** (która zaznacza obszar do zmiany) oraz opis, możesz zmienić część obrazu — na przykład dodać kapelusz naszemu zajączkowi.

```python
response = client.images.edit(
  model="gpt-image-1",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# zmiany są również zwracane jako base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Obraz bazowy zawiera tylko królika; ostateczny obraz ma kapelusz na króliku.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zastrzeżenie**:
Niniejszy dokument został przetłumaczony za pomocą usługi tłumaczenia AI [Co-op Translator](https://github.com/Azure/co-op-translator). Choć dążymy do dokładności, prosimy pamiętać, że automatyczne tłumaczenia mogą zawierać błędy lub niedokładności. Oryginalny dokument w jego języku źródłowym należy uznawać za autorytatywne źródło. W przypadku informacji krytycznych zalecane jest skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z użycia tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
